# Information Theory

## Learning Objectives
1. Implement entropy, cross-entropy, KL divergence, mutual information, and perplexity from first principles and verify key properties
2. Apply mutual information for feature selection and compare rankings against sklearn's implementation
3. Demonstrate the cross-entropy / softmax pipeline and its relationship to KL divergence in classification
4. Visualize the asymmetry of KL divergence and its practical implications for knowledge distillation and RLHF

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import make_classification

np.random.seed(42)
print("numpy:", np.__version__)
print("sklearn:", __import__('sklearn').__version__)
print("Setup complete.")

## Level 1: Five Core Information Theory Functions

Implement and verify all five core functions:
- H(X) = -sum p log p  (entropy, bits if log2)
- H(p,q) = -sum p log q  (cross-entropy)
- KL(P||Q) = sum P log P/Q  (KL divergence, asymmetric)
- I(X;Y) = H(X) - H(X|Y)  (mutual information)
- Perplexity = 2^H(X)  (language model evaluation)

Key verification: H(uniform K) = log2(K), H(deterministic) = 0

In [ ]:
# -----------------------------------------------------------------------
# Level 1: Five core information theory functions with verification
# -----------------------------------------------------------------------

def entropy(p: np.ndarray) -> float:
    """
    Shannon entropy in bits: H(X) = -sum p(x) * log2(p(x))

    Properties:
    - H(uniform) = log2(K) for K equally likely outcomes (maximum entropy)
    - H(deterministic) = 0 (no uncertainty)
    - H is always non-negative

    Parameters
    ----------
    p : probability distribution (must sum to 1, zeros handled correctly)

    Returns
    -------
    entropy in bits
    """
    p = p[p > 0]  # 0 * log(0) = 0 by convention; filtering avoids log(0)
    return -np.sum(p * np.log2(p))


def cross_entropy(p: np.ndarray, q: np.ndarray) -> float:
    """
    Cross-entropy: H(p, q) = -sum p(x) * log(q(x))

    Measures the expected bits needed to encode samples from p using codebook for q.
    H(p, q) >= H(p), with equality iff p == q.
    This is the loss function minimized in neural network classification.

    Parameters
    ----------
    p : true distribution (one-hot labels in classification)
    q : model distribution (softmax output); clipped to avoid log(0)

    Returns
    -------
    cross-entropy in nats (uses ln)
    """
    q = np.clip(q, 1e-10, 1)
    return -np.sum(p * np.log(q))


def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """
    KL divergence (relative entropy): KL(P || Q) = sum P(x) * log(P(x)/Q(x))

    Measures information lost when Q approximates P.
    Properties:
    - Always non-negative (Gibbs' inequality)
    - KL(P||Q) = 0 iff P = Q
    - ASYMMETRIC: KL(P||Q) != KL(Q||P) in general
    - Not a metric (violates triangle inequality)

    Parameters
    ----------
    p : true distribution P
    q : approximate distribution Q (clipped to avoid division by zero)

    Returns
    -------
    KL divergence in nats
    """
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))


def mutual_information(joint: np.ndarray) -> float:
    """
    Mutual information: I(X;Y) = H(X) + H(Y) - H(X,Y)
                                = sum_{x,y} p(x,y) * log(p(x,y) / (p(x)*p(y)))

    Measures statistical dependence: I(X;Y) = 0 iff X and Y are independent.
    Unlike correlation, captures ANY statistical dependence (linear and nonlinear).

    Parameters
    ----------
    joint : 2-D joint probability table p(x, y), must sum to 1

    Returns
    -------
    mutual information in nats
    """
    p_x = joint.sum(axis=1, keepdims=True)   # marginal P(X)
    p_y = joint.sum(axis=0, keepdims=True)   # marginal P(Y)
    product = p_x * p_y                       # product of marginals P(X)*P(Y)
    mask = joint > 0
    return np.sum(joint[mask] * np.log(joint[mask] / product[mask]))


def perplexity(log_probs: np.ndarray) -> float:
    """
    Perplexity = 2^H = exp(avg negative log probability)

    Interpretation: if PP = K, the model is as uncertain as a uniform distribution
    over K equally likely tokens at each step.
    Lower is better. Random LM over V-vocab has PP = V.

    Parameters
    ----------
    log_probs : array of log probabilities (base e) assigned to each token/word

    Returns
    -------
    perplexity (positive real number)
    """
    avg_neg_log2 = -np.mean(log_probs) / np.log(2)
    return 2 ** avg_neg_log2


# -----------------------------------------------------------------------
# Verification tests
# -----------------------------------------------------------------------

print("=== Entropy Verification ===")
K_vals = [2, 4, 8, 16, 32]
for K in K_vals:
    uniform = np.ones(K) / K
    h = entropy(uniform)
    expected = np.log2(K)
    status = "PASS" if abs(h - expected) < 1e-10 else "FAIL"
    print(f"  H(uniform K={K:2d}): {h:.4f}  expected log2({K})={expected:.4f}  [{status}]")

determ = np.array([1.0, 0.0, 0.0, 0.0])
h_det = entropy(determ)
print(f"  H(deterministic): {h_det:.4f}  expected 0.0000  [{'PASS' if h_det == 0 else 'FAIL'}]")

print()
print("=== Cross-Entropy >= Entropy (H(p,q) >= H(p)) ===")
p_true = np.array([0.7, 0.2, 0.1])
q_good = np.array([0.65, 0.25, 0.10])   # close to p
q_bad  = np.array([0.10, 0.80, 0.10])   # far from p
h_p    = -np.sum(p_true * np.log(p_true[p_true > 0]))
ce_good = cross_entropy(p_true, q_good)
ce_bad  = cross_entropy(p_true, q_bad)
kl_good = kl_divergence(p_true, q_good)
kl_bad  = kl_divergence(p_true, q_bad)
print(f"  H(p):       {h_p:.4f}")
print(f"  H(p,q_good):{ce_good:.4f}  KL={kl_good:.4f}  H(p)+KL={h_p+kl_good:.4f}  [PASS: CE=H+KL]")
print(f"  H(p,q_bad): {ce_bad:.4f}  KL={kl_bad:.4f}  H(p)+KL={h_p+kl_bad:.4f}")

print()
print("=== KL Asymmetry: KL(P||Q) != KL(Q||P) ===")
P = np.array([0.5, 0.4, 0.1])
Q = np.array([0.2, 0.3, 0.5])
kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)
print(f"  KL(P||Q) = {kl_pq:.4f}")
print(f"  KL(Q||P) = {kl_qp:.4f}  (different!)")
print(f"  Difference: {abs(kl_pq - kl_qp):.4f}")

print()
print("=== Mutual Information Verification ===")
# Independent: I(X;Y) should be 0
p_joint_indep = np.array([[0.3, 0.2], [0.15, 0.10], [0.15, 0.10]])
p_joint_indep = p_joint_indep / p_joint_indep.sum()  # normalize
mi_indep = mutual_information(p_joint_indep)
print(f"  MI (near-independent joint): {mi_indep:.6f}  (close to 0 = independent)")
# Dependent: I(X;Y) should be > 0
p_joint_dep = np.array([[0.45, 0.05], [0.05, 0.45]])
mi_dep = mutual_information(p_joint_dep)
print(f"  MI (strongly dependent):     {mi_dep:.4f}  (> 0 = dependent)")

print()
print("=== Perplexity Verification ===")
K = 100
# Perfect model: always predicts correct token with prob 1 -> PP = 1
log_probs_perfect = np.zeros(100)  # log(1) = 0
pp_perfect = perplexity(log_probs_perfect)
print(f"  PP (perfect model, log_p=0):    {pp_perfect:.2f}  (expected 1.0)")
# Random model over K=100 vocab: PP = 100
log_probs_random = np.full(100, np.log(1 / K))
pp_random = perplexity(log_probs_random)
print(f"  PP (random K={K} model):         {pp_random:.2f}  (expected {float(K):.1f})")
# Good language model: assign ~4% to correct token on average
log_probs_good = np.log(0.04) * np.ones(200) + np.random.normal(0, 0.1, 200)
pp_good = perplexity(log_probs_good)
print(f"  PP (good LM, avg correct=4%):   {pp_good:.1f}  (lower than K={K})")

## Level 2: Feature Selection via Mutual Information

Mutual information quantifies how much knowing a feature X reduces uncertainty about label Y.
It detects both linear and nonlinear relationships, unlike Pearson correlation.

We compare our manual MI implementation against sklearn's `mutual_info_classif` and
demonstrate that MI detects non-monotone relationships that correlation misses.

In [ ]:
# -----------------------------------------------------------------------
# Level 2: Feature selection via MI -- manual vs sklearn + nonlinear detection
# -----------------------------------------------------------------------

def manual_mi_continuous(X: np.ndarray, y: np.ndarray,
                          n_bins: int = 10) -> float:
    """
    Estimate mutual information between continuous feature X and discrete label y.

    Uses histogram binning to discretize X, then computes MI from joint distribution.
    Note: sklearn uses k-nearest neighbors which is more accurate; this is for illustration.

    Parameters
    ----------
    X      : 1-D continuous feature values
    y      : 1-D discrete labels
    n_bins : number of bins for discretizing X

    Returns
    -------
    estimated mutual information in nats
    """
    # Discretize X using quantile bins (more robust than equal-width)
    X_binned = np.digitize(X, np.percentile(X, np.linspace(0, 100, n_bins + 1)[1:-1]))
    n_x = n_bins
    n_y = len(np.unique(y))

    # Build joint distribution
    joint = np.zeros((n_x, n_y))
    y_int = (y - y.min()).astype(int)
    for xi, yi in zip(X_binned, y_int):
        xi_idx = min(xi, n_x - 1)
        joint[xi_idx, yi] += 1

    joint /= joint.sum()   # normalize to probabilities
    joint = np.maximum(joint, 0)

    return mutual_information(joint)


# Generate classification dataset with features of varying relevance
rng = np.random.default_rng(42)
n_samples = 2000
y_labels = rng.choice([0, 1], size=n_samples, p=[0.5, 0.5])

# Feature types
f1_linear    = y_labels * 2 + rng.normal(0, 1, n_samples)   # linear: high MI and correlation
f2_quadratic = (y_labels - 0.5) ** 2 + rng.normal(0, 0.1, n_samples)  # nonlinear (quadratic): MI>0, corr~0
f3_sine      = np.sin(y_labels * np.pi) + rng.normal(0, 0.3, n_samples)  # periodic: some MI
f4_noise     = rng.normal(0, 1, n_samples)  # pure noise: MI~0, corr~0
f5_threshold = (y_labels == 1).astype(float) + rng.normal(0, 0.2, n_samples)  # threshold: high MI

X_data = np.column_stack([f1_linear, f2_quadratic, f3_sine, f4_noise, f5_threshold])
feature_names = ["linear", "quadratic", "sine", "noise", "threshold"]

# sklearn MI
mi_sklearn = mutual_info_classif(X_data, y_labels, random_state=42)

# Manual MI (binning-based)
mi_manual = np.array([manual_mi_continuous(X_data[:, i], y_labels) for i in range(5)])

# Pearson correlation
correlations = np.array([abs(np.corrcoef(X_data[:, i], y_labels)[0, 1]) for i in range(5)])

print("Feature Selection Comparison: MI vs Pearson Correlation")
print(f"{'Feature':<12}  {'sklearn MI':>12}  {'Manual MI':>11}  {'|Pearson r|':>12}  Note")
print("-" * 70)
for i, name in enumerate(feature_names):
    note = ""
    if name == "quadratic":
        note = "<-- nonlinear: MI detects, corr misses"
    elif name == "noise":
        note = "<-- noise: both low"
    elif name == "linear":
        note = "<-- linear: both detect"
    print(f"{name:<12}  {mi_sklearn[i]:>12.4f}  {mi_manual[i]:>11.4f}  {correlations[i]:>12.4f}  {note}")

# Feature ranking comparison
rank_sklearn = np.argsort(-mi_sklearn) + 1
rank_corr    = np.argsort(-correlations) + 1
print()
print("Ranking agreement: sklearn MI vs Pearson correlation")
for i, name in enumerate(feature_names):
    agree = "agree" if rank_sklearn[i] == rank_corr[i] else f"DISAGREE (MI rank {rank_sklearn[i]}, corr rank {rank_corr[i]})"
    print(f"  {name:<12}: {agree}")

print()
print("Key insight: quadratic feature has near-zero correlation but substantial MI")
print("MI detects nonlinear dependencies that correlation misses entirely")

## Real-World Example 1: Cross-Entropy Loss in Neural Network Classification

Cross-entropy H(p,q) is the loss function for multi-class classification.
When p is one-hot labels and q is softmax outputs, minimizing CE is equivalent
to minimizing KL(P_data || Q_model) -- the model distribution approaches the data distribution.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 1: Softmax + cross-entropy pipeline
# -----------------------------------------------------------------------

def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """
    Numerically stable softmax with temperature scaling.

    Temperature T > 1: flatten distribution (higher entropy, more random)
    Temperature T < 1: sharpen distribution (lower entropy, more confident)
    Temperature T -> 0: argmax (deterministic, zero entropy)
    """
    z = logits / temperature
    z_shifted = z - z.max()   # subtract max for numerical stability
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum()


def cross_entropy_loss(logits: np.ndarray, true_class: int,
                        temperature: float = 1.0) -> dict:
    """
    Compute softmax probabilities and cross-entropy loss.

    For one-hot label p, CE = -log q[true_class] (only true class contributes).
    Gradient dCE/d_logit_k = q_k - p_k (elegantly simple).
    """
    probs = softmax(logits, temperature)
    loss = -np.log(probs[true_class])  # equivalent to cross_entropy with one-hot p

    # KL divergence interpretation
    p_onehot = np.zeros(len(logits))
    p_onehot[true_class] = 1.0
    h_p = 0.0  # H(one-hot) = 0
    kl = cross_entropy(p_onehot, probs) - h_p  # CE = H(p) + KL(P||Q) = 0 + KL

    gradients = probs.copy()
    gradients[true_class] -= 1.0   # dCE/d_logit = q - p

    return {"probs": probs, "loss": loss, "kl": kl, "gradients": gradients}


# Simulate training progression for a 4-class classifier
n_classes = 4
true_class = 2

print("Cross-Entropy Loss Training Progression (true class = 2)")
print(f"{'Step':<8}  {'Loss':>8}  {'KL':>8}  {'P(correct)':>12}  {'Entropy H(q)':>14}")
print("-" * 55)

# Simulate logits evolving during training
for step, logit_scale in enumerate([0.1, 0.5, 1.0, 2.0, 4.0, 8.0]):
    logits = np.array([-0.5, 0.2, logit_scale, 0.1])  # class 2 becomes increasingly confident
    result = cross_entropy_loss(logits, true_class)
    h_q = entropy(result["probs"])
    print(f"{step:<8d}  {result['loss']:>8.4f}  {result['kl']:>8.4f}  {result['probs'][true_class]:>12.4f}  {h_q:>14.4f}")

print()
print("Gradient (dCE/d_logit = q - p) at final step:")
for k, (grad, prob) in enumerate(zip(result["gradients"], result["probs"])):
    label = "<-- TRUE CLASS" if k == true_class else ""
    print(f"  Class {k}: grad={grad:+.4f}  prob={prob:.4f} {label}")

# Temperature scaling effect on entropy
print("
Temperature scaling effect on cross-entropy loss:")
logits_fixed = np.array([-1.0, 0.5, 2.0, 0.3])
print(f"{'Temp':>8}  {'Loss':>8}  {'P(correct)':>12}  {'H(q) bits':>12}  {'Distribution'}")
print("-" * 75)
for T in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    r = cross_entropy_loss(logits_fixed, 2, temperature=T)
    h_q = entropy(r["probs"])
    probs_str = " ".join([f"{p:.2f}" for p in r["probs"]])
    print(f"{T:>8.1f}  {r['loss']:>8.4f}  {r['probs'][2]:>12.4f}  {h_q:>12.4f}  [{probs_str}]")
print("T->0: argmax (sharp), T->inf: uniform (flat)")

## Real-World Example 2: KL Divergence in Knowledge Distillation

Knowledge distillation minimizes KL(teacher || student), encouraging the student
to match the teacher's soft probability distribution (not just the argmax).
Temperature scaling of teacher logits reveals soft target structure.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 2: KL divergence in knowledge distillation
# -----------------------------------------------------------------------

def distillation_loss(
    teacher_logits: np.ndarray,
    student_logits: np.ndarray,
    true_labels: np.ndarray,
    temperature: float = 3.0,
    alpha: float = 0.5
) -> dict:
    """
    Knowledge distillation loss: alpha * KL(teacher||student) + (1-alpha) * CE(hard)

    Teacher soft targets at temperature T reveal inter-class relationships
    (e.g. 'this image looks 70% like cat, 20% like dog').
    The student learns these soft relationships, not just hard labels.

    Parameters
    ----------
    temperature : T > 1 softens distributions, revealing class similarities
    alpha       : weight on soft (distillation) loss vs hard (supervised) loss

    Returns
    -------
    dict with loss components and analysis
    """
    n_classes = len(teacher_logits)

    # Soft probabilities at temperature T
    teacher_soft = softmax(teacher_logits, temperature=temperature)
    student_soft = softmax(student_logits, temperature=temperature)
    student_hard = softmax(student_logits, temperature=1.0)

    # Distillation loss: KL(teacher_soft || student_soft) * T^2
    # The T^2 factor compensates for the scaling of gradients at temperature T
    kl_soft = kl_divergence(teacher_soft, student_soft) * (temperature ** 2)

    # Hard label CE loss
    ce_hard = cross_entropy(true_labels, student_hard)

    # Combined loss
    total_loss = alpha * kl_soft + (1 - alpha) * ce_hard

    return {
        "kl_soft": kl_soft,
        "ce_hard": ce_hard,
        "total": total_loss,
        "teacher_soft": teacher_soft,
        "student_soft": student_soft,
    }


# Scenario: 5-class classifier, teacher is large model, student is small
# True class = 0
n_classes = 5
true_label = np.array([1.0, 0.0, 0.0, 0.0, 0.0])  # one-hot, class 0

# Teacher: confident on class 0, but assigns non-trivial prob to class 1 (similar class)
teacher_logits = np.array([4.0, 2.0, -1.0, -2.0, -3.0])

print("Knowledge Distillation: Effect of Temperature on Soft Targets")
print(f"
Teacher logits: {teacher_logits}")
print(f"
{'Temp':>6}  {'Teacher soft':>40}  {'KL(T||S)':>10}")
print("-" * 65)

# Fixed student logits (partially trained)
student_logits = np.array([3.0, 1.5, -0.5, -1.5, -2.5])

for T in [1.0, 2.0, 3.0, 5.0, 10.0]:
    result = distillation_loss(teacher_logits, student_logits, true_label,
                                temperature=T, alpha=0.5)
    tsoft_str = " ".join([f"{p:.3f}" for p in result["teacher_soft"]])
    print(f"{T:>6.1f}  [{tsoft_str}]  {result['kl_soft']:>10.4f}")

print()
print("At T=1: one-hot teacher (hard information only)")
print("At T=5: soft targets reveal class 1 is 'similar' to class 0")
print("High T -> KL can initially increase because more structure is revealed,")
print("then decreases as student learns the soft distribution")

print()
# Demonstrate: student improves by using soft targets
student_logits_list = [
    ("random student",    np.array([0.5, 0.3, 0.1, -0.2, -0.1])),
    ("partially trained", np.array([2.5, 1.8, -0.3, -1.0, -1.5])),
    ("well trained",      np.array([3.8, 1.9, -1.2, -1.9, -2.8])),
]
print("Student training progress (T=3, alpha=0.5):")
print(f"{'Student state':<20}  {'KL(T||S)':>10}  {'CE_hard':>10}  {'Total loss':>12}")
for name, slog in student_logits_list:
    r = distillation_loss(teacher_logits, slog, true_label, temperature=3.0, alpha=0.5)
    print(f"{name:<20}  {r['kl_soft']:>10.4f}  {r['ce_hard']:>10.4f}  {r['total']:>12.4f}")

## Real-World Example 3: Perplexity and Language Model Evaluation

Perplexity is the standard metric for language models.
PP = 2^H = exp(-mean_log_probability_of_tokens)

Comparing: a good LM, a random LM, and a bigram LM on a toy corpus.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 3 + Comparison: Perplexity and KL asymmetry visualization
# -----------------------------------------------------------------------

# ------ Perplexity comparison ------
vocab_size = 200
n_tokens = 500
rng = np.random.default_rng(42)

# True token probabilities (Zipf-like distribution -- realistic for language)
ranks = np.arange(1, vocab_size + 1)
p_true = 1.0 / (ranks ** 0.8)
p_true /= p_true.sum()

# Sample some tokens from true distribution
tokens = rng.choice(vocab_size, size=n_tokens, p=p_true)

# Good LM: assigns probability close to p_true
q_good = p_true * 0.9 + (1 - 0.9) / vocab_size   # slightly smoothed
q_good /= q_good.sum()
log_probs_good = np.log(q_good[tokens])

# Random LM: uniform distribution
q_random = np.ones(vocab_size) / vocab_size
log_probs_random = np.log(q_random[tokens])

# Bigram LM: rough approximation -- pairs of common tokens
q_bigram = p_true ** 0.5 / (p_true ** 0.5).sum()  # less peaked than true
log_probs_bigram = np.log(q_bigram[tokens])

# Oracle LM: exactly knows true distribution
log_probs_oracle = np.log(p_true[tokens])

pp_good   = perplexity(log_probs_good)
pp_random = perplexity(log_probs_random)
pp_bigram = perplexity(log_probs_bigram)
pp_oracle = perplexity(log_probs_oracle)

print("Language Model Perplexity Comparison")
print(f"{'Model':<20}  {'Perplexity':>12}  {'Mean log-prob':>14}  {'Interpretation'}")
print("-" * 70)
models_pp = [
    ("Oracle (true dist)",  pp_oracle,   np.mean(log_probs_oracle),  "Best possible"),
    ("Good LM (smoothed)",  pp_good,     np.mean(log_probs_good),    "Close to oracle"),
    ("Bigram LM",           pp_bigram,   np.mean(log_probs_bigram),  "Rough estimate"),
    ("Random (uniform)",    pp_random,   np.mean(log_probs_random),  f"PP = vocab_size = {vocab_size}"),
]
for name, pp, mlp, interp in models_pp:
    print(f"{name:<20}  {pp:>12.1f}  {mlp:>14.4f}  {interp}")

print()
print(f"Random baseline perplexity = vocab size = {vocab_size}")
print(f"Good LM is {pp_random/pp_good:.1f}x better than random (lower PP = better)")

# ------ KL asymmetry: forward vs reverse KL ------
print()
print("KL Asymmetry: Forward vs Reverse KL")
# P: bimodal distribution (two peaks)
x_grid = np.linspace(-5, 5, 200)
p_bimodal = (stats.norm.pdf(x_grid, -1.5, 0.5) + stats.norm.pdf(x_grid, 1.5, 0.5)) / 2
p_bimodal /= p_bimodal.sum()

# Q1: forward KL minimizer (broad, covers both modes)
q_forward = stats.norm.pdf(x_grid, 0, 1.5)
q_forward /= q_forward.sum()

# Q2: reverse KL minimizer (sharp, covers one mode)
q_reverse = stats.norm.pdf(x_grid, 1.5, 0.5)
q_reverse /= q_reverse.sum()

kl_fwd_q1 = kl_divergence(p_bimodal, q_forward)
kl_fwd_q2 = kl_divergence(p_bimodal, q_reverse)
kl_rev_q1 = kl_divergence(q_forward, p_bimodal)
kl_rev_q2 = kl_divergence(q_reverse, p_bimodal)

print(f"  Forward KL(P||Q):  broad Q={kl_fwd_q1:.4f},  narrow Q={kl_fwd_q2:.4f}  (broad is better)")
print(f"  Reverse KL(Q||P):  broad Q={kl_rev_q1:.4f},  narrow Q={kl_rev_q2:.4f}  (narrow is better)")
print("  Forward KL penalizes Q assigning low prob where P is high -> Q must cover ALL modes")
print("  Reverse KL penalizes Q assigning high prob where P is low -> Q concentrates on ONE mode")

# ------ Visualization ------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
ax1.plot(x_grid, p_bimodal / p_bimodal.max(), "k-", linewidth=2.5, label="P (bimodal truth)")
ax1.plot(x_grid, q_forward / q_forward.max(), "steelblue", linewidth=2, linestyle="--",
         label=f"Q forward (broad), KL(P||Q)={kl_fwd_q1:.3f}")
ax1.plot(x_grid, q_reverse / q_reverse.max(), "darkorange", linewidth=2, linestyle=":",
         label=f"Q reverse (narrow), KL(P||Q)={kl_fwd_q2:.3f}")
ax1.set_xlabel("x", fontsize=12)
ax1.set_ylabel("Normalized Density", fontsize=12)
ax1.set_title("KL Asymmetry: Forward vs Reverse KL", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2 = axes[1]
model_names_pp = ["Oracle", "Good LM", "Bigram", "Random"]
pp_vals = [pp_oracle, pp_good, pp_bigram, pp_random]
colors_pp = ["gold", "seagreen", "steelblue", "crimson"]
bars_pp = ax2.bar(model_names_pp, pp_vals, color=colors_pp, alpha=0.8, edgecolor="white")
ax2.axhline(vocab_size, color="red", linestyle="--", linewidth=1.5,
            label=f"Random baseline PP={vocab_size}")
for bar, pp_val in zip(bars_pp, pp_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{pp_val:.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylabel("Perplexity (lower = better)", fontsize=12)
ax2.set_title(f"Perplexity Comparison (vocab={vocab_size})", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("information_theory.png", dpi=100, bbox_inches="tight")
plt.show()

print("
Key takeaways:")
print("  H(p) = log2(K) for uniform, 0 for deterministic")
print("  CE = H(p) + KL(P||Q): minimizing CE is minimizing KL to data distribution")
print("  KL asymmetry determines mode-seeking vs mean-seeking behavior")
print("  MI detects nonlinear dependence; Pearson correlation misses it")
print("  Perplexity = 2^H: intuitive measure of LM quality (lower is better)")